# getattr

- 在 Python 中，getattr 是实现动态编程（反射）的核心工具。
- 它的基本语法是 getattr(对象, "属性或方法名字符串")。
- 当应用到模块时，它的作用就是根据一个字符串名字，从已经导入的模块中把对应的类、函数或变量“掏”出来。

结合你刚才提到的 verl 框架的代码场景，我们可以通过以下三个步骤来完整理解这个过程：

## 🛠️ 第一步：准备一个包含类的模块
假设我们有一个名为 engines.py 的文件，里面定义了几个推理引擎的类：

In [ ]:
# engines.py
class VLLMEngine:
    def generate(self):
        return "使用 vLLM 生成文本..."

class SGLangEngine:
    def generate(self):
        return "使用 SGLang 生成文本..."

## 🔍 第二步：动态导入模块并使用 getattr 提取类
- 在另一个主程序文件中，我们不需要提前知道具体要用哪个引擎，而是通过字符串来动态获取：

In [ ]:
import importlib

# 1. 动态导入模块（等同于 import engines）
module_name = "engines"
engines_module = importlib.import_module(module_name)

# 2. 假设我们从配置文件中读取到了想要的类名（字符串形式）
target_class_name = "VLLMEngine" 

# 3. 使用 getattr 从模块中“提取”出真正的类对象
# 这行代码等同于直接写： EngineClass = engines.VLLMEngine
EngineClass = getattr(engines_module, target_class_name)

print(EngineClass) 
# 输出: <class 'engines.VLLMEngine'> 
# 此时 EngineClass 已经是一个真正的 Python 类了！

## 🚀 第三步：实例化并调用（返回给调用者）
既然 getattr 已经帮我们拿到了真正的类对象，接下来就可以像平常一样去实例化它，并返回给框架的其他部分使用：

In [ ]:
# 4. 实例化这个类
engine_instance = EngineClass()

# 5. 调用它的方法
print(engine_instance.generate()) 
# 输出: 使用 vLLM 生成文本...

## 💡 为什么要这么做？（结合 verl 框架的理解）
在 verl 框架的 get_rollout_class 函数中，return getattr(rollout_module, class_name) 之所以直接返回类对象而不是实例，是因为框架的设计采用了工厂模式：
1. 框架的调用者拿到这个类后，可以自己在合适的时机、传入自定义的参数去创建实例（例如：MyClass = get_rollout_class(...); instance = MyClass(config=...)）。
2.<font color='red'> 这种“动态导入 + getattr 提取”的方式，让框架具备了极强的扩展性。</font>未来如果增加了新的推理引擎（比如 lmdeploy），只需要在注册表里加一行字符串映射，框架就能自动识别并加载，完全不需要修改底层的导入逻辑。